# Camada Bronze - Ingestão e Armazenamento dos Dados

## Objetivo

A camada Bronze é responsável pela ingestão dos dados provenientes das APIs públicas do Banco Central do Brasil e pelo armazenamento dos dados em seu formato original, preservando a estrutura disponibilizada pela fonte.

Esta camada representa o primeiro estágio da arquitetura Medallion e serve como base para os tratamentos e transformações realizados nas camadas posteriores.

## Estrutura Inicial

Criação dos schemas utilizados na arquitetura Medallion:

- `bronze` → Dados brutos
- `silver` → Dados tratados
- `gold` → Dados analíticos

## Fontes de Dados

#### Estatísticas de Meios de Pagamento

Dados obtidos através da API pública do Banco Central contendo informações estatísticas sobre meios de pagamento no Brasil.

#### Transações PIX por Município

Dados obtidos através da API pública do Banco Central contendo informações sobre transações PIX segregadas por município, estado e região.

## Processamento Realizado

- Consumo das APIs utilizando módulo de ingestão no diretório /src (`ingestion.py`);
- Conversão dos dados retornados para DataFrames Spark;
- Armazenamento dos dados brutos sem aplicação de regras de negócio;
- Persistência dos dados em formato Delta Lake.

## Tabelas Geradas

Após a ingestão, os dados são armazenados nas seguintes tabelas:

- `bronze.pix_transactions`
- `bronze.payment_statistics`

## Benefícios da Camada Bronze

- Preserva os dados originais da fonte;
- Permite rastreabilidade e auditoria dos dados;
- Facilita reprocessamentos futuros;
- Separa a etapa de ingestão das etapas de transformação;
- Implementa as boas práticas da arquitetura Medallion.

In [0]:
%sql
--Criação dos Schemas Arquitetura medalhão
CREATE SCHEMA IF NOT EXISTS bronze;
CREATE SCHEMA IF NOT EXISTS silver;
CREATE SCHEMA IF NOT EXISTS gold;

In [0]:
#Import da pasta src com arquivo de injestão de dados
import sys
sys.path.append("/Workspace/DataEngineering/Analytics/src")

from ingestion import get_data

#Chamada da função de captura de dados da api, retornando as estatísticas de pagamentos
payment = get_data(
    f"https://olinda.bcb.gov.br/olinda/servico/MPV_DadosAbertos/versao/v1/odata/MeiosdePagamentosMensalDA(AnoMes=@AnoMes)?"
    f"@AnoMes='202001'&$format=json"
)
df_payment = spark.createDataFrame(payment)

#Chamada da função de captura de dados da api, retornando informações do sobre o PIX
pix = get_data(
    f"https://olinda.bcb.gov.br/olinda/servico/Pix_DadosAbertos/versao/v1/odata/TransacoesPixPorMunicipio(DataBase=@DataBase)?"
    f"@DataBase='202001'&$format=json"
)
df_pix = spark.createDataFrame(pix)

#Criação das tabelas delta com modo de reescrita
df_pix.write.format("delta").mode("overwrite").saveAsTable("bronze.pix_transactions")
print("Tabela bronze.pix_transactions criada")
df_payment.write.format("delta").mode("overwrite").saveAsTable("bronze.payment_statistics")
print("Tabela bronze.payment_statistics criada")
